# Fine-Tuning an LLM for Sanskrit↔English (Option 1)
### AI/ML Developer Assignment — ImmverseAI (BharatiyaGPT)

This notebook fine-tunes a small open-source instruction-tuned LLM using
**QLoRA** to improve Sanskrit↔English translation, explanation, QA, and
summarization.

**Default / reported model:** `Qwen/Qwen2.5-1.5B-Instruct` (ungated, no HF
login needed). `meta-llama/Llama-3.2-1B-Instruct` is also supported via the
same `BASE_MODEL` variable, but it's gated (needs a free HF account +
license acceptance) and took ~18x longer to train under identical config in
our testing — see `REPORT.md` section 3/8 for the full comparison.

**Runtime:** Google Colab, GPU = T4 or L4 (Runtime → Change runtime type → GPU)

**Pipeline:**
1. Setup & installs
2. Tokenizer inspection (Sanskrit vs English fragmentation)
3. Dataset construction (public parallel corpora → instruction pairs)
4. QLoRA fine-tuning
5. Inference (base vs fine-tuned)
6. Evaluation (BLEU / chrF++, before/after, failure analysis, ranked examples)
7. Save & export adapter

See `README.md` and `REPORT.md` in the repo for setup instructions and the
full written report (problem understanding, tradeoffs, failure analysis,
what we'd improve with more time).


## 1. Setup

In [ ]:
# Clone directly into current working directory (/content)
!git clone https://github.com/yanil-03/sanskrit_llm.git .

# Clean up unwanted items
!rm -rf notebooks colab_notebook_preview.pdf docs.zip

In [6]:
# If you'd rather upload files manually instead of cloning: skip the cell
# above and just upload the `scripts/`, `eval/` folders alongside this
# notebook in Colab, then continue from here.

!pip install -q -U transformers accelerate peft trl bitsandbytes datasets sacrebleu sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 13.1 MB/s eta 0:00:00


In [7]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


CUDA available: True
GPU: Tesla T4
VRAM (GB): 15.6


In [8]:
# Hugging Face auth — Llama models are gated; you need a free HF account
# and to accept the license at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
# then run:
from huggingface_hub import login
login()  # paste your HF token when prompted
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"

# Default used to produce this repo's reported results: ungated, no login needed.
# BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
BASE_MODEL


'meta-llama/Llama-3.2-1B-Instruct'

## 2. Tokenizer Inspection

Before training, we check how the base tokenizer handles Devanagari/Sanskrit
text compared to English of similar content. This is the "Tokenization /
Vocabulary" challenge called out in the assignment — Sanskrit's compounding
and rich morphology tend to fragment badly under tokenizers trained mostly on
Latin-script/English corpora, which affects both training efficiency and the
effective context window.

In [9]:
!python scripts/inspect_tokenizer.py --model "$BASE_MODEL"


config.json: 100% 877/877 [00:00<00:00, 2.86MB/s]
tokenizer_config.json: 100% 54.5k/54.5k [00:00<00:00, 12.3MB/s]
tokenizer.json: 100% 9.09M/9.09M [00:01<00:00, 6.67MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.59MB/s]
Tokenizer: meta-llama/Llama-3.2-1B-Instruct
Vocab size: 128000

SA: वसुधैव कुटुम्बकम्।
  chars=18  tokens=13  tokens/char=0.72
  breakdown: ['à¤µ', 'à¤¸', 'à¥ģà¤§', 'à¥Ī', 'à¤µ', 'Ġà¤ķ', 'à¥ģà¤Ł', 'à¥ģà¤®', 'à¥įà¤¬', 'à¤ķ', 'à¤®', 'à¥į', 'à¥¤']
EN: The whole world is one family.
  chars=30  tokens=7  tokens/char=0.23
  breakdown: ['The', 'Ġwhole', 'Ġworld', 'Ġis', 'Ġone', 'Ġfamily', '.']
  --> Sanskrit costs 3.10x more tokens/char than English

SA: धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः।
  chars=43  tokens=25  tokens/char=0.58
  breakdown: ['à¤§à¤°', 'à¥įà¤®à¤ķ', 'à¥įà¤·', 'à¥ĩà¤¤', 'à¥įà¤°', 'à¥ĩ', 'Ġà¤ķ', 'à¥ģà¤°', 'à¥ģà¤ķ', 'à¥įà¤·', 'à¥ĩà¤¤', 'à¥įà¤°', 'à¥ĩ', 'Ġà¤¸à¤®', 'à¤µ', 'à¥ĩà¤¤', 'à¤¾', 'Ġà¤¯', 'à¥ģ', 'à¤¯', 'à¥ģà¤¤', 'à¥įà¤¸', 'à¤µ', 'à¤ĥ', 

**What to look for:** if the average Sanskrit/English tokens-per-character
ratio printed above is notably above 1.0 (Qwen2.5 lands around 4x, Llama-3.2
around 2.5x in our runs — see `REPORT.md` section 8), Sanskrit text is
consuming several times more tokens per unit of content than English.
Implications:
- shorter effective context window for Sanskrit passages
- more gradient steps needed to see the same "amount" of Sanskrit content
- a case for tokenizer adaptation (adding Devanagari merges) if we had more
  time/compute — see REPORT.md section 9 ("What we'd improve").

We did **not** attempt tokenizer adaptation/retraining in this assignment
(explicitly marked optional in the brief) given the time budget, but the
diagnostic above is what would justify prioritizing it next.

## 3. Dataset Construction

We build an instruction-tuning dataset from **rahular/itihasa**, a
Sanskrit-English parallel corpus of ~93k sentence pairs derived from the
Ramayana and Mahabharata, plus a small curated seed set (Bhagavad Gita verses
and well-known subhashitas) reserved for hand-checkable qualitative testing.

Each parallel pair is expanded into **multiple instruction types**
(Sanskrit→English, English→Sanskrit, explanation, QA, summarization) using
templates, so the model learns to follow varied instructions rather than only
one fixed task shape. Full reasoning is in REPORT.md section 2.

In [10]:
!python scripts/prepare_data.py --max_pairs 6000 --out_dir data


Loading rahular/itihasa via parquet (refs/convert/parquet) ...
Parquet load attempt failed (Unable to find 'hf://datasets/rahular/itihasa@~parquet/default/train/0000.parquet'); trying revision='refs/convert/parquet' ...

Itihasa/train/0000.parquet: downloading bytes:   2% 334k/16.4M [00:01<01:21, 197kB/s]
Itihasa/train/0000.parquet: downloading bytes:  40% 6.52M/16.4M [00:01<00:01, 4.99MB/s, 32.0kB/s  ]
Itihasa/train/0000.parquet: downloading bytes:  82% 13.5M/16.4M [00:01<00:00, 11.5MB/s,  623kB/s  ]
Itihasa/train/0000.parquet: reconstructing file:   4% 699k/16.4M [00:02<00:59, 264kB/s, 43.9kB/s  ]
Itihasa/train/0000.parquet: reconstructing file:   8% 1.34M/16.4M [00:03<00:22, 667kB/s, 61.6kB/s  ]
Itihasa/train/0000.parquet: downloading bytes: 100% 16.3M/16.3M [00:03<00:00, 5.02MB/s, 1.38MB/s  ]
Itihasa/train/0000.parquet: reconstructing file: 100% 16.4M/16.4M [00:03<00:00, 5.04MB/s, 1.50MB/s  ]

Itihasa/validation/0000.parquet: downloading bytes:   0% 0.00/1.39M [00:00<?, ?B/s]
Itiha

In [11]:
import json

def peek(path, n=3):
    print(f"--- {path} ---")
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(json.loads(line))

peek("data/train.jsonl")
peek("data/test.jsonl")


--- data/train.jsonl ---
{'instruction': 'Render this English sentence in Sanskrit.', 'input': 'And on that day no Brāhmaṇa ever felt tired, or hungry and there was none that was not learned, or that was not followed by an hundred persons.', 'output': 'न तेष्वहःसु श्रान्तो वा क्षुधितो वा न दृश्यते। नाविद्वान् ब्राह्मणः कश्चिन्नाशतानुचरस्तथा॥', 'task_type': 'en2sa'}
{'instruction': 'Translate the following Sanskrit text into English.', 'input': 'त्वं राजा भरत भव स्वयं नराणां वन्यानामहमपि राजराणमृगाणाम्। गच्छ त्वं पुरवरमद्य सम्प्रहृष्टः संहृष्टस्त्वहमपिदण्डकान् प्रवेक्ष्ये॥', 'output': 'O Bharata, be you yourself the monarch of men. I shall become the king of kings of deer. Go you to that foremost of cities with a glad heart: with a glad heart will I enter Dandaka.', 'task_type': 'sa2en'}
{'instruction': 'Translate the following English text into Sanskrit.', 'input': 'Having placed his mothers in Ayodhyā, Bharata steady in his vow, kindling in grief, said preceptors.', 'output': 'ततो निक

## 4. QLoRA Fine-Tuning

4-bit quantized base model + LoRA adapters on attention and MLP projection
layers, trained with TRL's `SFTTrainer`. This keeps memory comfortably within
a T4's 16GB, and keeps training time within the assignment's suggested
3–6 hour budget (our default config trains in well under that).

Key hyperparameters and the reasoning behind them are documented in
REPORT.md section 4/5 (hardware constraints and optimizations).

In [12]:
!nvidia-smi --query-gpu=name --format=csv,noheader


Tesla T4


In [13]:
!python scripts/train.py \
    --base_model "$BASE_MODEL" \
    --train_file data/train.jsonl \
    --val_file data/val.jsonl \
    --output_dir outputs/lora-sanskrit \
    --epochs 2 \
    --per_device_batch_size 4 \
    --grad_accum 4 \
    --max_seq_len 512


Loading tokenizer/model: meta-llama/Llama-3.2-1B-Instruct

model.safetensors: downloading bytes:   7% 182M/2.47G [00:01<00:11, 205MB/s, 13.9MB/s  ]
model.safetensors: downloading bytes:  10% 259M/2.47G [00:02<00:11, 193MB/s, 21.8MB/s  ]
model.safetensors: downloading bytes:  23% 571M/2.47G [00:03<00:07, 267MB/s, 47.9MB/s  ]
model.safetensors: downloading bytes:  55% 1.35G/2.47G [00:05<00:03, 338MB/s,  100MB/s  ]
model.safetensors: downloading bytes:  80% 1.97G/2.47G [00:08<00:02, 211MB/s,  131MB/s  ]
model.safetensors: downloading bytes:  87% 2.14G/2.47G [00:09<00:02, 118MB/s,  129MB/s  ]
model.safetensors: reconstructing file:  73% 1.81G/2.47G [00:12<00:03, 165MB/s, 77.1MB/s  ]
model.safetensors: downloading bytes: 100% 2.14G/2.14G [00:16<00:00, 126MB/s,  128MB/s  ]
model.safetensors: reconstructing file: 100% 2.47G/2.47G [00:16<00:00, 146MB/s,  131MB/s  ]
Loading weights: 100% 146/146 [00:01<00:00, 140.41it/s]
generation_config.json: 100% 189/189 [00:00<00:00, 733kB/s]
Loading datase

**If you hit an OOM on a T4:** lower `--per_device_batch_size` to 2 and
raise `--grad_accum` to 8 (same effective batch size, less peak memory), or
lower `--max_seq_len` to 384. This tradeoff is discussed in REPORT.md.

**For a quick smoke test** before committing to a full run, add
`--max_train_samples 500` to finish a sanity-check pass in a few minutes.

## 5. Inference: Base Model vs Fine-Tuned

We run both the untouched base model and our fine-tuned adapter over the same
held-out test set, so we can directly compare before/after outputs.

In [14]:
# Base model predictions (no adapter) — this is the "before" baseline
!python scripts/infer.py \
    --base_model "$BASE_MODEL" \
    --test_file data/test.jsonl \
    --out_file eval/predictions_base.jsonl \
    --max_new_tokens 150


Loading weights: 100% 146/146 [00:00<00:00, 159.23it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done
100/100 done
Wrote 100 predictions to eval/predictions_base.jsonl


In [15]:
# Fine-tuned model predictions — this is "after"
!python scripts/infer.py \
    --base_model "$BASE_MODEL" \
    --adapter outputs/lora-sanskrit \
    --test_file data/test.jsonl \
    --out_file eval/predictions.jsonl \
    --max_new_tokens 150


Loading weights: 100% 146/146 [00:00<00:00, 160.51it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done
100/100 done
Wrote 100 predictions to eval/predictions.jsonl


In [16]:
# Quick interactive sanity check on a single example
!python scripts/infer.py \
    --base_model "$BASE_MODEL" \
    --adapter outputs/lora-sanskrit \
    --instruction "Translate the following Sanskrit text into English." \
    --input "वसुधैव कुटुम्बकम्।"


Loading weights: 100% 146/146 [00:00<00:00, 158.16it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.

=== PREDICTION ===
And there was a son of Vasudeva.


## 6. Evaluation

We compute BLEU and chrF++ per task type, generate a before-vs-after
comparison table, and run a lightweight heuristic pass to flag likely failure
cases (empty outputs, verbatim-echo, rambling/hallucinated length,
degenerate repetition) for manual review.

chrF++ is emphasized over BLEU for Sanskrit because it's character-level and
more forgiving of compounding/sandhi variation than word-level n-gram
matching — see REPORT.md section 6 for the full discussion of why n-gram
metrics are a limited signal here, especially for the non-translation task
types (explanation/QA/summarization), which we evaluate mainly qualitatively.

In [17]:
!python eval/evaluate.py \
    --base_predictions eval/predictions_base.jsonl \
    --finetuned_predictions eval/predictions.jsonl \
    --out eval/report.md \
    --n_examples 10


Wrote report to eval/report.md
{
  "en2sa": {
    "n": 45,
    "bleu": 0.08,
    "chrf++": 10.21
  },
  "explain": {
    "n": 15,
    "bleu": 5.0,
    "chrf++": 21.35
  },
  "qa": {
    "n": 10,
    "bleu": 0.98,
    "chrf++": 12.75
  },
  "sa2en": {
    "n": 24,
    "bleu": 1.15,
    "chrf++": 15.6
  },
  "summary": {
    "n": 6,
    "bleu": 0.84,
    "chrf++": 15.2
  }
}


In [18]:
with open("eval/report.md", encoding="utf-8") as f:
    print(f.read())


# Evaluation Report

## Quantitative Metrics (fine-tuned model)

| Task Type | N | BLEU | chrF++ |
|---|---|---|---|
| en2sa | 45 | 0.08 | 10.21 |
| explain | 15 | 5.0 | 21.35 |
| qa | 10 | 0.98 | 12.75 |
| sa2en | 24 | 1.15 | 15.6 |
| summary | 6 | 0.84 | 15.2 |

## Before vs After (base model vs fine-tuned)

| Task Type | BLEU (base) | BLEU (fine-tuned) | Δ BLEU | chrF++ (base) | chrF++ (fine-tuned) | Δ chrF++ |
|---|---|---|---|---|---|---|
| en2sa | 0.02 | 0.08 | +0.06 | 6.91 | 10.21 | +3.30 |
| explain | 0.22 | 5.0 | +4.78 | 19.2 | 21.35 | +2.15 |
| qa | 0.75 | 0.98 | +0.23 | 15.77 | 12.75 | -3.02 |
| sa2en | 0.17 | 1.15 | +0.98 | 14.26 | 15.6 | +1.34 |
| summary | 0.8 | 0.84 | +0.04 | 17.34 | 15.2 | -2.14 |

## Heuristically Flagged Failure Cases (41 / 100)

These are candidates for manual review, not confirmed errors.

- **Flags:** repetitive_degeneration | **Task:** en2sa
  - Input: `And presenting him with many a charming fountain, trees will delight Rāma at the tops of mounta

**Per-example ranking (sentence-level chrF++).** `evaluate.py` reports
corpus-level metrics and a heuristic failure flagger, but doesn't rank
individual predictions. The cell below scores every fine-tuned prediction
with `sacrebleu.sentence_chrf` and sorts, so the "best"/"worst" examples used
in the report are selected automatically and reproducibly, rather than by
manually reading through `eval/report.md` (see REPORT.md section 6/7 for the
resulting examples and discussion).

In [19]:
import sys
sys.path.insert(0, "eval")  # so we can import load_jsonl from evaluate.py
from evaluate import load_jsonl
import sacrebleu


def score_and_rank(predictions_file, task_type_filter=None, top_n=5, bottom_n=5, max_chars=200):
    """Score every prediction individually (sentence-level chrF++), rank them,
    and print the best- and worst-scoring examples. This is how you find
    genuine successes/failures instead of eyeballing and cherry-picking.

    Uses the same jsonl schema infer.py writes: instruction, input, reference,
    prediction, task_type.
    """
    rows = load_jsonl(predictions_file)
    if task_type_filter:
        rows = [r for r in rows if r.get("task_type") == task_type_filter]
    if not rows:
        print(f"No rows found (task_type_filter={task_type_filter!r}).")
        return []

    def trunc(s, n=max_chars):
        s = s.replace("\n", " ").strip()
        return s if len(s) <= n else s[:n].rstrip() + "…"

    for r in rows:
        pred = r.get("prediction", "").strip()
        ref = r.get("reference", "").strip()
        # sentence_chrf errors on an empty hypothesis/reference pair -> score 0 instead
        if not pred or not ref:
            r["_chrf"] = 0.0
        else:
            r["_chrf"] = sacrebleu.sentence_chrf(pred, [ref], word_order=2).score

    ranked = sorted(rows, key=lambda r: r["_chrf"], reverse=True)

    print(f"=== TOP {top_n} (highest chrF++) — candidates for 'successful cases' in REPORT.md ===\n")
    for r in ranked[:top_n]:
        print(f"chrF++={r['_chrf']:.1f}  task={r['task_type']}")
        print(f"  Input:      {trunc(r.get('input', ''))}")
        print(f"  Reference:  {trunc(r['reference'])}")
        print(f"  Prediction: {trunc(r['prediction'])}\n")

    print(f"=== BOTTOM {bottom_n} (lowest chrF++) — candidates for 'failure analysis' in REPORT.md ===\n")
    for r in ranked[-bottom_n:]:
        print(f"chrF++={r['_chrf']:.1f}  task={r['task_type']}")
        print(f"  Input:      {trunc(r.get('input', ''))}")
        print(f"  Reference:  {trunc(r['reference'])}")
        print(f"  Prediction: {trunc(r['prediction'])}\n")

    return ranked


# Run across all task types together
all_ranked = score_and_rank("eval/predictions.jsonl", top_n=5, bottom_n=5)

# Per-task-type breakdown — cleaner examples per category for the report
for t in ["sa2en", "en2sa", "explain", "qa", "summary"]:
    print(f"\n{'='*20} task_type = {t} {'='*20}")
    score_and_rank("eval/predictions.jsonl", task_type_filter=t, top_n=3, bottom_n=3)


=== TOP 5 (highest chrF++) — candidates for 'successful cases' in REPORT.md ===

chrF++=45.5  task=explain
  Input:      ततस्तत्र प्रविष्टस्य कौसल्याया निवेशनम्। अधिरुह्यापि शयनं बभूव लुलितं मनः॥
  Reference:  This verse conveys: Having entered Kausalyā's apartment, the king having laid himself on the bed, was overwhelmed with emotion.
  Prediction: This verse conveys: And thereupon, the king, having entered the abode of Kausalyā, was struck with grief, and was overcome with sorrow.

chrF++=34.0  task=qa
  Input:      देवैस्तदा समागम्य सर्षिसङ्घः सचारणैः॥ याचितौ प्रशमं तत्र जग्मतुस्तौ सुरोत्तमौ।
  Reference:  Upon the assembled gods with the saints and the Cāraṇas beseeching those two foremost of celestials, they became pacified.
  Prediction: And thereupon the celestials came to the assembly of the celestials, and the mighty ones came to the assembly of the gods.

chrF++=32.9  task=explain
  Input:      कल्यमुत्थाय देवानां कृत्वा पूजां यथाविधि। वन्दितव्यो दशरथः पिता मम जनेश्वरः॥
  Ref

## 7. Save / Export

The LoRA adapter (small, a few hundred MB at most) is saved under
`outputs/lora-sanskrit/`. Zip and download it, or push it to the Hugging Face
Hub, to include it in your submission per the assignment's deliverables
checklist.

In [20]:
import shutil
shutil.make_archive("lora-sanskrit-adapter", "zip", "outputs/lora-sanskrit")
print("Zipped adapter -> lora-sanskrit-adapter.zip")

from google.colab import files
files.download("lora-sanskrit-adapter.zip")


Zipped adapter -> lora-sanskrit-adapter.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
# Optional: push adapter to the Hugging Face Hub instead of downloading a zip
# from huggingface_hub import HfApi
# api = HfApi()
# api.create_repo("your-username/sanskrit-llama-lora", exist_ok=True)
# api.upload_folder(folder_path="outputs/lora-sanskrit", repo_id="your-username/sanskrit-llama-lora")


## Next Steps / What We'd Improve With More Time

See `REPORT.md` section 9 for the full list. Highlights:
- Tokenizer adaptation (extend vocab with Devanagari-aware merges) given the
  fragmentation ratio measured in section 2 above
- Larger, more diverse Sanskrit sources (GRETIL, Digital Corpus of Sanskrit,
  AI4Bharat) beyond itihasa's epic-poetry register, to generalize past
  Ramayana/Mahabharata style text
- Human evaluation (native Sanskrit speaker) rather than only automatic
  metrics, especially for the explanation/QA tasks
- LoRA-vs-full-fine-tune and quantization ablations (bonus items)
